In [ ]:
# --- Configuration -----------------------------------------------------------
# Path to the folder containing the CSV data files, and where to save figures.
#
# Default layout assumes a local repository:
#     ./data/      CSV files
#     ./figures/   output SVGs
#
# To run in Google Colab with Drive, uncomment the two lines below and point
# DATA_DIR / OUTPUT_DIR to your Drive folders.

# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = "data"        # e.g. "/content/drive/My Drive/pdata"
OUTPUT_DIR = "figures"   # e.g. "/content/drive/My Drive/pdata/fixed"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Define the parameter grid for RandomForest
param_grid = {
    'max_depth': [82],
    'min_samples_leaf': [1],
    'min_samples_split': [3],
    'n_estimators': [78],
    'max_features': ['log2'],
    'bootstrap': [False]
}

#PSO method

In [ ]:
import numpy as np
import pandas as pd
from math import sqrt
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load your data
se = pd.read_csv(f"{DATA_DIR}/final_data_standardized.csv")

# Define the features and target
features =['Kappa-3 (Fast Descriptors)', 'Molecular density (Spatial Descriptors)', 'PEOE_VSA6(RDkit)', 'SMR_VSA10(RDKit)']
target = 'pIC50'

#KennardStone
train = se.drop(labels=[17, 18, 28, 29, 30, 31, 33, 37, 39, 50] + list(range(54, 76)))
test = se.loc[[17, 18, 28, 29, 30, 31, 33, 37, 39, 50]]

# Train-test split
X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

# Initialize RandomForest model
model = RandomForestRegressor(random_state=42)

# Use GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(estimator=model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='r2')

# Fit the grid search on the training data
grid_search.fit(X_train, y_train)

# Get the best model from GridSearchCV
best_model = grid_search.best_estimator_

# Output the best parameters
print("\nBest Parameters Found:")
print(grid_search.best_params_)

# Train set predictions
y_pred_train = best_model.predict(X_train)
r2_train = r2_score(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
rmse_train = sqrt(mse_train)

# Test set predictions
y_pred_test = best_model.predict(X_test)
r2_test = r2_score(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = sqrt(mse_test)

# Create DataFrames for train & test results
train_results = pd.DataFrame({'Actual': y_train, 'Predicted': y_pred_train, 'Error': y_train - y_pred_train})
test_results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_test, 'Error': y_test - y_pred_test})

# Output model performance in separate lines
print("\nModel Performance:")
print(f"R² Train: {r2_train:.4f}")
print(f"RMSE Train: {rmse_train:.4f}")
print(f"R² Test: {r2_test:.4f}")
print(f"RMSE Test: {rmse_test:.4f}")

# Print Train and Test Results
print("\nTrain Set Results:")
print(train_results.to_string(index=False))  # Prevents long index printing

print("\nTest Set Results:")
print(test_results.to_string(index=False))  # Prevents long index printing


In [ ]:
!pip install optuna


In [ ]:
# After LOOCV, you can use the final best model on your test set
X_test = test[features]
y_test = test[target]

# Fit the final model to the entire training data and predict the test set
best_model.fit(X_train, y_train)
y_pred_test = best_model.predict(X_test)

# Evaluate the model on the training set
y_pred_train = best_model.predict(X_train)

train_results = pd.DataFrame({'Actual': y_train, 'Predicted': y_pred_train, 'Error': y_train - y_pred_train})
test_results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_test, 'Error': y_test - y_pred_test})

print("Train Set Results:")
print(train_results)
print("Test Set Results:")
print(test_results)


# Calculate Train RMSE and Train R²
train_rmse = sqrt(mean_squared_error(y_train, y_pred_train))
train_r2 = r2_score(y_train, y_pred_train)

# Print Train RMSE and Train R²
print(f"Train RMSE: {train_rmse:.3f}")
print(f"Train R²: {train_r2:.3f}")

# Evaluate the model on the test set
test_rmse = sqrt(mean_squared_error(y_test, y_pred_test))
test_r2 = r2_score(y_test, y_pred_test)

# ---- Y-Randomization ---- #
from sklearn.utils import shuffle

# Shuffle the target values (y_train) to randomize them
y_train_randomized = shuffle(y_train, random_state=42)

# Fit the model again using randomized target values
best_model.fit(X_train, y_train_randomized)
y_pred_train_randomized = best_model.predict(X_train)

# Calculate RMSE for the randomized model
r2_randomized = r2_score(y_train, y_pred_train_randomized)

# Print test set evaluation
print(f"Test RMSE: {test_rmse:.3f}")
print(f"Test R²: {test_r2:.3f}")


# Print Y-Randomization results
print(f'Y-Randomized Train R²: {r2_randomized:.3f}')

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have the predictions for both train and test sets
train_Y = y_train
train_pred = y_pred_train
test_Y = y_test
test_pred = y_pred_test

plt.figure(figsize=(11, 5))

# 1 row, 2 column, plot 1 (SVR model predictions)
plt.subplot(1, 2, 1)
plt.scatter(x=train_Y, y=train_pred, alpha=0.6, label='Training Set')
plt.scatter(x=test_Y, y=test_pred, c="Red", alpha=0.4, label='Testing Set')
#plt.scatter(y_external, y_pred_external, color="green", alpha=0.6, label='External Validation Set')
plt.plot([6.0, 8.5], [6.0, 8.5], "--", color="Black")

plt.legend(loc="lower right")
plt.grid(linewidth=0.2, alpha=0.6)
plt.title('RF model')
plt.ylabel('Prediction')
plt.xlabel('Experiment')
plt.savefig(f"{OUTPUT_DIR}/RF_GFA_Kappa3.svg", format='svg')
# Adjust layout and display the plot
plt.tight_layout()
plt.show()

# Optionally save the plot as a .jpg file


In [ ]:
import numpy as np
import pandas as pd
from math import sqrt
from sklearn.utils import shuffle
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load your data
se = pd.read_csv(f"{DATA_DIR}/final_data_standardized.csv")

# Define the features and target
features =['Kappa-3 (Fast Descriptors)', 'Molecular density (Spatial Descriptors)', 'PEOE_VSA6(RDkit)', 'SMR_VSA10(RDKit)']
target = 'pIC50'

#KennardStone
train = se.drop(labels=[17, 18, 28, 29, 30, 31, 33, 37, 39, 50] + list(range(54, 76)))
test = se.loc[[17, 18, 28, 29, 30, 31, 33, 37, 39, 50]]

# Train-test split
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Define hyperparameter grid
param_grid = {
    'max_depth': [82],
    'min_samples_leaf': [1],
    'min_samples_split': [3],
    'n_estimators': [78],
    'max_features': ['log2'],
    'bootstrap': [False]
}
# Number of iterations
n_iterations = 10
results_all = []

for i in range(n_iterations):
    print(f"\nIteration {i+1}/{n_iterations}")

    # Initialize RandomForest model
    model = RandomForestRegressor(bootstrap=True, random_state=i)

    # GridSearchCV for hyperparameter tuning
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        n_jobs=-1,
        verbose=0,
        scoring='r2'
    )
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    # Train set predictions
    y_pred_train = best_model.predict(X_train)
    train_rmse = sqrt(mean_squared_error(y_train, y_pred_train))
    train_r2 = r2_score(y_train, y_pred_train)

    # Test set predictions
    y_pred_test = best_model.predict(X_test)
    test_rmse = sqrt(mean_squared_error(y_test, y_pred_test))
    test_r2 = r2_score(y_test, y_pred_test)

    # Y-Randomization
    y_train_randomized = shuffle(y_train, random_state=i)
    best_model.fit(X_train, y_train_randomized)
    y_pred_train_randomized = best_model.predict(X_train)
    r2_randomized = r2_score(y_train, y_pred_train_randomized)

    # Store iteration results
    iteration_results = {
        'Iteration': i + 1,
        'Train RMSE': train_rmse,
        'Train R²': train_r2,
        'Test RMSE': test_rmse,
        'Test R²': test_r2,
        'Y-Randomized Train R²': r2_randomized,
        'Best Parameters': grid_search.best_params_
    }
    results_all.append(iteration_results)

    # Print iteration results
    print(f"Train R²: {train_r2:.4f} | Train RMSE: {train_rmse:.4f}")
    print(f"Test R²: {test_r2:.4f} | Test RMSE: {test_rmse:.4f}")
    print(f"Y-Randomized Train R²: {r2_randomized:.4f}")
    print("-" * 50)

# Calculate average results
average_results = {
    metric: np.mean([result[metric] for result in results_all])
    for metric in ['Train RMSE', 'Train R²', 'Test RMSE', 'Test R²', 'Y-Randomized Train R²']
}

# Print averaged results
print("\nAverage Results Over All Iterations:")
for key, value in average_results.items():
    print(f"{key}: {value:.4f}")

# Create DataFrames for Train & Test Results
train_results = pd.DataFrame({'Actual': y_train, 'Predicted': y_pred_train, 'Error': y_train - y_pred_train})
test_results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_test, 'Error': y_test - y_pred_test})

# Print Train and Test Results
print("\nTrain Set Results:")
print(train_results.to_string(index=False))

print("\nTest Set Results:")
print(test_results.to_string(index=False))


In [ ]:
# ---- Y-Randomization ---- #
r2_randomized_list = []
for i in range(30):
    # Shuffle the target values (y_train) to randomize them
    y_train_randomized = shuffle(y_train, random_state=i) # Use different random_state for each iteration

    # Fit the model again using randomized target values
    best_model.fit(X_train, y_train_randomized)
    y_pred_train_randomized = best_model.predict(X_train)

    # Calculate RMSE for the randomized model
    r2_randomized = r2_score(y_train, y_pred_train_randomized)
    r2_randomized_list.append(r2_randomized)

    # Print Y-Randomization results for each iteration
    print(f'Y-Randomized Train R² (Iteration {i+1}): {r2_randomized:.3f}')

# Now r2_randomized_list contains R-squared values for all 30 iterations.
# You can analyze this list to assess the model's performance with shuffled targets.
print(f"List of R2 values from Y-Randomization: {r2_randomized_list}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import shap
import matplotlib.pyplot as plt

# Load your data
se = pd.read_csv(f"{DATA_DIR}/final_data_standardized.csv")

# Define the features and target
features =['Kappa-3 (Fast Descriptors)', 'Molecular density (Spatial Descriptors)', 'PEOE_VSA6(RDkit)', 'SMR_VSA10(RDKit)']
target = 'pIC50'

#KennardStone
train = se.drop(labels=[17, 18, 28, 29, 30, 31, 33, 37, 39, 50] + list(range(54, 76)))
test = se.loc[[17, 18, 28, 29, 30, 31, 33, 37, 39, 50]]

# Train-test split
X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

# Train the Random Forest model
model = RandomForestRegressor(
    n_estimators=82,
    max_depth=78,
    min_samples_split=3,
    min_samples_leaf=1,
    max_features='log2',
    bootstrap=False
)
model.fit(X_train, y_train)

# Create SHAP explainer and calculate SHAP values
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_train)

# Custom SHAP scatter plot for each feature
#output_dir = f"{OUTPUT_DIR}/shap_plots/"  # Specify folder in Google Drive
#import os
#os.makedirs(output_dir, exist_ok=True)  # Create directory if it doesn't exist

for i, feature in enumerate(features):
    plt.figure(figsize=(8, 6))

    # Scatter plot of SHAP values vs feature values
    plt.scatter(X_train[feature], shap_values.values[:, i], color='black', alpha=0.7)

    # Add horizontal and vertical threshold lines
    plt.axhline(y=0, color='black', linestyle='--')  # SHAP value = 0
    plt.axvline(x=0, color='black', linestyle='--')  # Feature value = 0 (adjust if you have a different threshold)

    # Titles and labels
    plt.title(f"SHAP Value for {feature}", fontsize=12)
    plt.xlabel(feature, fontsize=12)
    plt.ylabel("SHAP Value", fontsize=12)
    plt.grid(False)  # Optional: remove grid if unnecessary

    # Save plot to Google Drive as an SVG file
    #save_path = os.path.join(output_dir, f'shap_scatter_BEST_{feature}.svg')
    #plt.savefig(save_path, format='svg', bbox_inches='tight')

    # Show the plot
    plt.show()

#print(f"Plots saved to {output_dir}")


In [ ]:
# prompt: there are compounds 55-64 that are newly designed, use that as new test set create the code

# Define the new test set indices
new_test_indices = list(range(54, 76))  # Compounds 55-64 (inclusive)

# Create the new test set
new_test = se.loc[new_test_indices]
X_new_test = new_test[features]
y_new_test = new_test[target]

# Predict using the best model
y_pred_new_test = best_model.predict(X_new_test)

# Evaluate the new test set
r2_new_test = r2_score(y_new_test, y_pred_new_test)
mse_new_test = mean_squared_error(y_new_test, y_pred_new_test)
rmse_new_test = sqrt(mse_new_test)

# Create a DataFrame for the new test results
new_test_results = pd.DataFrame({'Actual': y_new_test, 'Predicted': y_pred_new_test, 'Error': y_new_test - y_pred_new_test})

# Output model performance on the new test set
print("\nModel Performance on New Test Set:")
print(f"R² New Test: {r2_new_test:.4f}")
print(f"RMSE New Test: {rmse_new_test:.4f}")

# Print New Test Results
print("\nNew Test Set Results:")
print(new_test_results.to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd
import optuna
from math import sqrt
from sklearn.utils import shuffle
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

# Load data
se = pd.read_csv(f"{DATA_DIR}/cut_outlier.csv")

# Define features and target
features = ['VSA_EState9(RDkit)', 'Dipole y (VAMP Electrostatics)', 'AlogP98 (Fast Descriptors)', 'E-state keys (sums): S_aasC (Fast Descriptors)']
target = 'pIC50'

# Kennard-Stone split
train = se.drop(labels=[17, 18, 28, 29, 30, 31, 33, 37, 39, 50])
test = se.loc[[17, 18, 28, 29, 30, 31, 33, 37, 39, 50]]

# Train-test split
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Define Optuna objective function
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 10, 100),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
    }

    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)  # Optimize on test set

    return r2_score(y_test, y_pred_test)  # Use test R² instead of CV R²

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, n_jobs=-1)


# Get the best hyperparameters
best_params = study.best_params
print("\nBest Hyperparameters Found:", best_params)

# Train the best model using the best hyperparameters
best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)

# Train set predictions
y_pred_train = best_model.predict(X_train)
train_rmse = sqrt(mean_squared_error(y_train, y_pred_train))
train_r2 = r2_score(y_train, y_pred_train)

# Test set predictions
y_pred_test = best_model.predict(X_test)
test_rmse = sqrt(mean_squared_error(y_test, y_pred_test))
test_r2 = r2_score(y_test, y_pred_test)

# Y-Randomization
y_train_randomized = shuffle(y_train, random_state=42)
best_model.fit(X_train, y_train_randomized)
y_pred_train_randomized = best_model.predict(X_train)
r2_randomized = r2_score(y_train, y_pred_train_randomized)

# Print final results
print("\nFinal Model Performance:")
print(f"Train R²: {train_r2:.4f} | Train RMSE: {train_rmse:.4f}")
print(f"Test R²: {test_r2:.4f} | Test RMSE: {test_rmse:.4f}")
print(f"Y-Randomized Train R²: {r2_randomized:.4f}")

# Create DataFrames for Train & Test Results
train_results = pd.DataFrame({'Actual': y_train, 'Predicted': y_pred_train, 'Error': y_train - y_pred_train})
test_results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_test, 'Error': y_test - y_pred_test})

# Print Train and Test Results
print("\nTrain Set Results:")
print(train_results.to_string(index=False))

print("\nTest Set Results:")
print(test_results.to_string(index=False))


###SVR

In [ ]:
from math import sqrt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

# Load your data
# Load your data
se = pd.read_csv(f"{DATA_DIR}/final_data_standardized.csv")

# Define the features and target
features =['Kappa-3 (Fast Descriptors)', 'Molecular density (Spatial Descriptors)', 'PEOE_VSA6(RDkit)', 'SMR_VSA10(RDKit)']
target = 'pIC50'

#KennardStone
train = se.drop(labels=[17, 18, 28, 29, 30, 31, 33, 37, 39, 50] + list(range(54, 76)))
test = se.loc[[17, 18, 28, 29, 30, 31, 33, 37, 39, 50]]

# Separating the target (Y) and features (X) for the training set
X_train = train[features]
y_train = train[target]




In [ ]:
# Define the parameter grid for SVR
#Adjust to results
# {'C': [390.26561574619393], 'epsilon': [0.1273811996516116], 'kernel': ['rbf']}

param_grid_svr = {'C': [10],
    'degree': [2],# Adjusted for both underfitting and overfitting scenarios
    'epsilon': [0.1],  # Adjusted range
    'kernel': ['rbf'],
    'gamma':[1]# Focus on one kernel for simplicity, or adjust based on results
}



In [ ]:
def objective_svr(trial):
    params = {
        'C': trial.suggest_loguniform("C", 0.1, 1000),
        'epsilon': trial.suggest_loguniform("epsilon", 0.01, 1.0),
        'kernel': trial.suggest_categorical("kernel", ["linear", "rbf"])
    }
    model = SVR(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_train)
    return mean_squared_error(y_train, y_pred, squared=False)  # RMSE

# Run Optuna optimization for SVR
study_svr = optuna.create_study(direction="minimize")
study_svr.optimize(objective_svr, n_trials=50)

best_params_svr = study_svr.best_params
best_model_svr = SVR(**best_params_svr)
best_model_svr.fit(X_train, y_train)
y_train_pred_svr = best_model_svr.predict(X_train)
r2_svr = r2_score(y_train, y_train_pred_svr)
rmse_svr = mean_squared_error(y_train, y_train_pred_svr, squared=False)

print("Best parameters for SVR: ", best_params_svr)
print("SVR R² Train: ", r2_svr, "RMSE Train: ", rmse_svr)


In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV


# Initialize the SVR model
svr_model = SVR()

# Initialize GridSearchCV for SVR
grid_search_svr = GridSearchCV(estimator=svr_model, param_grid=param_grid_svr,
                               cv=5, n_jobs=-1, verbose=2, scoring="neg_root_mean_squared_error")

# Fit the grid search on the training data
grid_search_svr.fit(X_train, y_train)

# Get the best model from the grid search
best_model_svr = grid_search_svr.best_estimator_

# Output the best parameters
print("Best parameters for SVR: ", grid_search_svr.best_params_)



In [ ]:
# After LOOCV, you can use the final best model on your test set
X_test = test[features]
y_test = test[target]
# Fit the final model to the entire training data and predict the test set
best_model_svr.fit(X_train, y_train)
y_pred_test = best_model_svr.predict(X_test)

# Evaluate the model on the training set
y_pred_train = best_model_svr.predict(X_train)

train_results = pd.DataFrame({'Actual': y_train, 'Predicted': y_pred_train, 'Error': y_train - y_pred_train})
test_results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_test, 'Error': y_test - y_pred_test})

print("Train Set Results:")
print(train_results)
print("Test Set Results:")
print(test_results)


# Calculate Train RMSE and Train R²
train_rmse_svr = sqrt(mean_squared_error(y_train, y_pred_train))
train_r2_svr = r2_score(y_train, y_pred_train)

# Print Train RMSE and Train R²
print(f"Train RMSE: {train_rmse_svr:.3f}")
print(f"Train R²: {train_r2_svr:.3f}")

# Evaluate the model on the test set
test_rmse_svr = sqrt(mean_squared_error(y_test, y_pred_test))
test_r2_svr = r2_score(y_test, y_pred_test)

# ---- Y-Randomization ---- #
from sklearn.utils import shuffle

# Shuffle the target values (y_train) to randomize them
y_train_randomized_svr = shuffle(y_train, random_state=42)

# Fit the model again using randomized target values
best_model_svr.fit(X_train, y_train_randomized_svr)
y_pred_train_randomized_svr = best_model_svr.predict(X_train)

# Calculate RMSE and R² for the randomized model
r2_randomized_svr = r2_score(y_train, y_pred_train_randomized_svr)

# Print test set evaluation
print(f"Test RMSE: {test_rmse_svr:.3f}")
print(f"Test R²: {test_r2_svr:.3f}")

# Print Y-Randomization results
print(f'Y-Randomized Train R²: {r2_randomized_svr:.3f}')


In [ ]:
import matplotlib.pyplot as plt

# Assuming you have the predictions for both train and test sets
train_Y = y_train
train_pred = y_pred_train
test_Y = y_test
test_pred = y_pred_test

plt.figure(figsize=(11, 5))

# 1 row, 2 column, plot 1 (SVR model predictions)
plt.subplot(1, 2, 1)
plt.scatter(x=train_Y, y=train_pred, alpha=0.6, label='Training Set')
plt.scatter(x=test_Y, y=test_pred, c="Red", alpha=0.4, label='Testing Set')
#plt.scatter(y_external, y_pred_external, color="green", alpha=0.6, label='External Validation Set')
plt.plot([6.0, 8.5], [6.0, 8.5], "--", color="Black")

plt.legend(loc="lower right")
plt.grid(linewidth=0.2, alpha=0.6)
plt.title('SVR Model')
plt.ylabel('Prediction')
plt.xlabel('Experiment')

plt.savefig(f"{OUTPUT_DIR}/SVR_GFA_Kappa3.svg", format='svg')

# Adjust layout and display the plot
plt.tight_layout()
plt.show()

# Optionally save the plot as a .jpg file
# plt.savefig(f"{OUTPUT_DIR}/SVR_Predictions.jpg", format='jpg')


###XGB

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
# prompt: install sklearn 1.2.2

!pip install scikit-learn==1.2.2

In [ ]:
from math import sqrt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

# Load your data
se = pd.read_csv(f"{DATA_DIR}/final_data_standardized.csv")

# Define the features and target
features =['Kappa-3 (Fast Descriptors)', 'Molecular density (Spatial Descriptors)', 'PEOE_VSA6(RDkit)', 'SMR_VSA10(RDKit)']
target = 'pIC50'

#KennardStone
train = se.drop(labels=[17, 18, 28, 29, 30, 31, 33, 37, 39, 50] + list(range(54, 76)))
test = se.loc[[17, 18, 28, 29, 30, 31, 33, 37, 39, 50]]

# Separating the target (Y) and features (X) for the training set
X_train = train[features]
y_train = train[target]




In [ ]:
import optuna
import xgboost as xgb
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error

se = pd.read_csv(f"{DATA_DIR}/cut_outlier.csv")

# Define the features and target
features = ['VSA_EState9(RDkit)', 'Dipole y (VAMP Electrostatics)', 'AlogP98 (Fast Descriptors)', 'E-state keys (sums): S_aasC (Fast Descriptors)']
target = 'pIC50'

# Train-test split based on Kennard-Stone selection
train = se.drop(labels=[17, 18, 28, 29, 30, 31, 33, 37, 39, 50])
test = se.loc[[17, 18, 28, 29, 30, 31, 33, 37, 39, 50]]

# Separating the target (Y) and features (X) for the training set
X_train = train[features]
y_train = train[target]

def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int("max_depth", 3, 20),
        'n_estimators': trial.suggest_int("n_estimators", 50, 500),
        'learning_rate': trial.suggest_loguniform("learning_rate", 0.01, 0.3),
        'colsample_bytree': trial.suggest_uniform("colsample_bytree", 0.5, 1.0),
    }
    model = xgb.XGBRegressor(**params, objective='reg:squarederror')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_train)
    return mean_squared_error(y_train, y_pred, squared=False)  # RMSE

# Run Optuna optimization for XGB
study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=50)

best_params_xgb = study_xgb.best_params
best_model_xgb = xgb.XGBRegressor(**best_params_xgb, objective='reg:squarederror')
best_model_xgb.fit(X_train, y_train)
y_train_pred_xgb = best_model_xgb.predict(X_train)
r2_xgb = r2_score(y_train, y_train_pred_xgb)
rmse_xgb = mean_squared_error(y_train, y_train_pred_xgb, squared=False)

print("Best parameters for XGBoost: ", best_params_xgb)
print("XGBoost R² Train: ", r2_xgb, "RMSE Train: ", rmse_xgb)


In [ ]:
# Define the parameter grid for XGBoost
param_grid_xgb = {
    'max_depth': [10],
    'n_estimators': [50],
    'learning_rate': [0.05],
    'colsample_bytree': [0.5],
    #'subsample':[0.9]# Random subset of features to consider for each tree
}

import xgboost as xgb
from sklearn.model_selection import GridSearchCV


# Initialize the XGBoost Regressor model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror')

# Initialize GridSearchCV for XGBoost
grid_search_xgb = GridSearchCV(estimator=xgb_model, param_grid=param_grid_xgb,
                               cv=5, n_jobs=-1, verbose=2, scoring="neg_root_mean_squared_error")

# Fit the grid search on the training data
grid_search_xgb.fit(X_train, y_train)

# Get the best model from the grid search
best_model_xgb = grid_search_xgb.best_estimator_

# Output the best parameters
print("Best parameters for XGBoost: ", grid_search_xgb.best_params_)



In [ ]:
# After LOOCV, you can use the final best model on your test set
X_test = test[features]
y_test = test[target]

# Fit the final model to the entire training data and predict the test set
best_model_xgb.fit(X_train, y_train)
y_pred_test = best_model_xgb.predict(X_test)

# Evaluate the model on the training set
y_pred_train = best_model_xgb.predict(X_train)

train_results = pd.DataFrame({'Actual': y_train, 'Predicted': y_pred_train, 'Error': y_train - y_pred_train})
test_results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_test, 'Error': y_test - y_pred_test})

print("Train Set Results:")
print(train_results)
print("Test Set Results:")
print(test_results)

# Calculate Train RMSE and Train R²
train_rmse_xgb = sqrt(mean_squared_error(y_train, y_pred_train))
train_r2_xgb = r2_score(y_train, y_pred_train)

# Print Train RMSE and Train R²
print(f"Train RMSE: {train_rmse_xgb:.3f}")
print(f"Train R²: {train_r2_xgb:.3f}")

# Evaluate the model on the test set
test_rmse_xgb = sqrt(mean_squared_error(y_test, y_pred_test))
test_r2_xgb = r2_score(y_test, y_pred_test)

# ---- Y-Randomization ---- #
from sklearn.utils import shuffle

# Shuffle the target values (y_train) to randomize them
y_train_randomized_xgb = shuffle(y_train, random_state=42)

# Fit the model again using randomized target values
best_model_xgb.fit(X_train, y_train_randomized_xgb)
y_pred_train_randomized_xgb = best_model_xgb.predict(X_train)

# Calculate RMSE and R² for the randomized model
r2_randomized_xgb = r2_score(y_train, y_pred_train_randomized_xgb)

# Print test set evaluation
print(f"Test RMSE: {test_rmse_xgb:.3f}")
print(f"Test R²: {test_r2_xgb:.3f}")

# Print Y-Randomization results
print(f'Y-Randomized Train R²: {r2_randomized_xgb:.3f}')



In [ ]:
import matplotlib.pyplot as plt

# Assuming `best_model_xgb` is trained and predictions (`train_pred_xgb`, `test_pred_xgb`) have been made
plt.figure(figsize=(11, 5))

# 1 row, 2 column, plot 1
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_pred_train, alpha=0.6, label="Training set")
plt.scatter(y_test, y_pred_test, c="Red", alpha=0.4, label="Testing set")
#plt.scatter(y_external, y_pred_external, color="green", alpha=0.6, label='External Validation Set')
plt.plot([6.0, 8.5], [6.0, 8.5], "--", color="Black")  # Reference diagonal line
plt.legend(loc="lower right")
plt.grid(linewidth=0.2, alpha=0.6)
plt.title('XGBoost Model')
plt.ylabel('Prediction')
plt.xlabel('Experiment')
plt.savefig(f"{OUTPUT_DIR}/XGB_GFA_Kappa3.svg", format='svg')
# Adjust layout and display the plot
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

# Mount Google Drive

# Load dataset
se = pd.read_csv(f"{DATA_DIR}/cut_outlier.csv")

# Define features and target
features = ['VSA_EState9(RDkit)', 'Dipole y (VAMP Electrostatics)', 'AlogP98 (Fast Descriptors)', 'E-state keys (sums): S_aasC (Fast Descriptors)']
target = 'pIC50'

# Kennard-Stone split
train = se.drop(labels=[17, 18, 28, 29, 30, 31, 33, 37, 39, 50])
test = se.loc[[17, 18, 28, 29, 30, 31, 33, 37, 39, 50]]

# Separating features and target for training
X_train = train[features]
y_train = train[target]

# Define XGBoost parameters
param_grid_xgb = {'colsample_bytree': 0.6957500898928296, 'learning_rate': 0.12585306702234647, 'max_depth': 6, 'n_estimators': 67}

# Train XGBoost model
model_xgb = xgb.XGBRegressor(
    objective='reg:squarederror',
    max_depth=param_grid_xgb['max_depth'],
    n_estimators=param_grid_xgb['n_estimators'],
    learning_rate=param_grid_xgb['learning_rate'],
    colsample_bytree=param_grid_xgb['colsample_bytree']
)

model_xgb.fit(X_train, y_train)

# Create SHAP explainer
explainer_xgb = shap.Explainer(model_xgb, X_train)

# Calculate SHAP values
shap_values_xgb = explainer_xgb(X_train)

# Generate SHAP summary plot
shap.summary_plot(shap_values_xgb, X_train, feature_names=features, plot_type='dot', show=False)

# Customize the title
plt.title("SHAP Summary Plot for XGBoost Model")

# Save the plot as SVG
#output_dir = f"{OUTPUT_DIR}/shap_plots/"
#os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists
#plt.savefig(f'{output_dir}/shap_summary_plot_xgb.svg', format='svg', bbox_inches='tight')

# Display confirmation
#print(f"SHAP summary plot saved to {output_dir}")


In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import os

# Ensure the output directory exists
#output_dir = f"{OUTPUT_DIR}/shap_plots_xgb/"
#os.makedirs(output_dir, exist_ok=True)

# Calculate SHAP values for the best XGBoost model
explainer_xgb = shap.Explainer(best_model_xgb, X_train)
shap_values_xgb = explainer_xgb(X_train)

# Custom SHAP scatter plot for each feature
for i, feature in enumerate(features):
    plt.figure(figsize=(8, 6))

    # Scatter plot of SHAP values vs feature values
    plt.scatter(X_train[feature], shap_values_xgb.values[:, i], color='black', alpha=0.7)

    # Add horizontal and vertical threshold lines
    plt.axhline(y=0, color='black', linestyle='--')  # SHAP value = 0
    plt.axvline(x=0, color='black', linestyle='--')  # Feature value = 0 (adjust if you have a different threshold)

    # Titles and labels
    plt.title(f"SHAP Value for {feature} (XGBoost)", fontsize=12)
    plt.xlabel(feature, fontsize=12)
    plt.ylabel("SHAP Value", fontsize=12)
    plt.grid(False)  # Optional: remove grid if unnecessary

    # Save plot to Google Drive as an SVG file
    #save_path = os.path.join(output_dir, f'shap_scatter_xgb_{feature}.svg')
    #plt.savefig(save_path, format='svg', bbox_inches='tight')

    # Show the plot
    plt.show()

print(f"SHAP scatter plots for XGBoost saved to {output_dir}")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Data for pIC50 values
data = {
    "Compound": list(range(1, 55)),
    "pIC50": [
        6.68, 6.66, 7.06, 7.26, 7.47, 7.34, 7.69, 7.73, 6.77,
        6.67, 7.66, 6.38, 7.85, 7.82, 7.75, 6.73, 7.22, 7.34, 7.34,
        7.46, 7.18, 7.8, 7.38, 7.77, 7.57, 7.67, 7.51, 7.96, 7.84,
        7.79, 7.95, 7.46, 7.43, 7.96, 7.63, 7.89, 6.71, 6.71, 7.21,
        7.38, 6.7, 6.89, 6.85, 6.86, 7.72, 6.62, 6.81, 6.88, 7.13,
        7.07, 6.73, 6.58, 6.78, 6.59
    ]
}

# Create a DataFrame
df = pd.DataFrame(data)

# Define custom bin edges from 6.30 to 7.95 with step size of 0.15
bins = np.arange(6.30, 8.10, 0.15)  # 8.10 ensures 7.95 is included

# Create histogram
plt.figure(figsize=(10, 6))
plt.hist(df['pIC50'], bins=bins, color='black', edgecolor='white')

# Customize the plot
plt.title(r'pIC$_{50}$ Distribution of α-ketoamide derivatives', fontsize=14, fontweight='bold')
plt.xlabel(r'Range of activity: pIC$_{50}$', fontsize=12)
plt.ylabel('Amount of compounds', fontsize=12)
plt.xticks(bins, rotation=45)  # Rotate labels for better visibility

# Save the plot
plt.savefig(f"{OUTPUT_DIR}/distribution.svg", format='svg')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
import shap

# Mount Google Drive

# Load dataset
se = pd.read_csv(f"{DATA_DIR}/cut_outlier.csv")

# Define features and target
features = ['VSA_EState9(RDkit)', 'Dipole y (VAMP Electrostatics)',
            'AlogP98 (Fast Descriptors)', 'E-state keys (sums): S_aasC (Fast Descriptors)']
target = 'pIC50'

# Kennard-Stone split
train = se.drop(labels=[17, 18, 28, 29, 30, 31, 33, 37, 39, 50])
test = se.loc[[17, 18, 28, 29, 30, 31, 33, 37, 39, 50]]

# Separating features and target for training
X_train = train[features]
y_train = train[target]

# Compute the correlation matrix
correlation_matrix = X_train.corr()

# Ensure diagonal is 1.00
np.fill_diagonal(correlation_matrix.values, 1)

# Create a **lower triangular mask**
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool), k=1)

# Set plot aesthetics
plt.figure(figsize=(4, 3), dpi=150)  # Adjusted size
sns.set_theme(style="white")

# Create heatmap with a cool-toned palette
heatmap = sns.heatmap(correlation_matrix, annot=True, cmap="Blues", fmt=".2f",
                      linewidths=1.2, mask=mask, vmin=-1, vmax=1,
                      cbar_kws={"shrink": 0.8}, square=True)

# **Fix text alignment**
heatmap.set_xticklabels(heatmap.get_xticklabels(), fontsize=6, weight='bold', rotation=90, ha="center")
heatmap.set_yticklabels(heatmap.get_yticklabels(), fontsize=6, weight='bold', rotation=0)

# Reduce annotation font size
for text in heatmap.texts:
    text.set_fontsize(5)

# Set title
plt.title("Correlation Matrix", fontsize=8, weight='bold', pad=5)

# **Save as SVG file**
plt.savefig(f"{OUTPUT_DIR}/correlation_matrix_VIF.svg", format="svg", bbox_inches="tight")

plt.show()


In [ ]:
# prompt: save the graph

# Assuming you want to save the last generated plot (the correlation matrix)
plt.savefig(f"{OUTPUT_DIR}/correlation_matrix_GFA.svg", format='svg')
